# LeetCode 1827 — Minimum Operations to Make the Array Increasing bằng Python

Link đề: https://leetcode.com/problems/minimum-operations-to-make-the-array-increasing/

Độ khó: **Easy** · Chủ đề: Array, Greedy

Skeleton chữ ký LeetCode:

```python
class Solution:
    def minOperations(self, nums: List[int]) -> int:

```

**Mục tiêu học của notebook này**

1. Hiểu chính xác "tăng **nghiêm ngặt**" (strictly increasing) khác "không giảm" ra sao.
2. Ôn các công cụ Python cơ bản: `list`, chỉ số, `range(1, n)`, `max()`, `+=`, sao chép list bằng `nums[:]`.
3. Đi từ cách mô phỏng từng phép `+1` (chậm) đến lời giải **tham lam** (greedy) một lượt `O(n)`.
4. Biết thêm một góc nhìn sáng tạo: công thức **prefix max của `nums[j] - j`**.
5. Tự trace, tự chứng minh vì sao tham lam cho số phép **ít nhất**, rồi luyện qua 6 bài tập.

## 1. Bài toán yêu cầu gì?

Cho mảng số nguyên `nums` (đánh chỉ số từ 0). Mỗi **phép toán** (operation): chọn **một** phần tử và **tăng nó thêm 1**.

Trả về **số phép ít nhất** để `nums` trở thành **tăng nghiêm ngặt**: `nums[i] < nums[i+1]` với mọi `0 <= i < len(nums) - 1`. Mảng 1 phần tử coi như đã tăng nghiêm ngặt.

| `nums` | Kết quả | Giải thích ngắn |
|---|---:|---|
| `[1,1,1]` | `3` | thành `[1,2,3]`: phần tử 2 tăng 1, phần tử 3 tăng 2 |
| `[1,5,2,4,1]` | `14` | thành `[1,5,6,7,8]`: tăng 0+0+4+3+7 |
| `[8]` | `0` | một phần tử, không cần làm gì |

**Constraints**: `1 <= nums.length <= 5000`, `1 <= nums[i] <= 10^4`.

Chú ý hai điều quan trọng:

- Ta **chỉ được tăng**, không được giảm. Vì thế phần tử đầu tiên không bao giờ cần đổi.
- "Nghiêm ngặt" nghĩa là **không được bằng nhau**: `[1,2,2]` **chưa** đạt.

In [1]:
from typing import List

examples = [([1, 1, 1], 3), ([1, 5, 2, 4, 1], 14), ([8], 0)]
for nums, expected in examples:
    print("nums =", nums, "-> expected =", expected)

nums = [1, 1, 1] -> expected = 3
nums = [1, 5, 2, 4, 1] -> expected = 14
nums = [8] -> expected = 0


## 2. Nền tảng Python cần biết

### 2.1. `list` và chỉ số

`list` là dãy có thứ tự. `nums[0]` là phần tử đầu, `nums[-1]` là phần tử cuối, `len(nums)` là số phần tử. Gán `nums[i] = v` sẽ **sửa** phần tử tại chỗ.

In [2]:
nums = [1, 5, 2]
print("len =", len(nums))
print("nums[0] =", nums[0], "| nums[-1] =", nums[-1])
nums[2] = 6
print("sau khi gán nums[2] = 6:", nums)

len = 3
nums[0] = 1 | nums[-1] = 2
sau khi gán nums[2] = 6: [1, 5, 6]


### 2.2. `range(1, n)` để so sánh với phần tử đứng trước

Bài này luôn so sánh `nums[i]` với `nums[i-1]`. Nếu bắt đầu từ `i = 0` thì `nums[-1]` lại là **phần tử cuối** (Python không báo lỗi!), gây sai ngầm. Vì vậy ta duyệt `i` từ `1`.

In [3]:
nums = [1, 5, 2, 4]
for i in range(1, len(nums)):
    print(f"i={i}: so sánh nums[{i-1}]={nums[i-1]} với nums[{i}]={nums[i]}")
print("Cẩn thận: nums[-1] =", nums[-1], "(phần tử cuối, KHÔNG phải 'trước phần tử 0')")

i=1: so sánh nums[0]=1 với nums[1]=5
i=2: so sánh nums[1]=5 với nums[2]=2
i=3: so sánh nums[2]=2 với nums[3]=4
Cẩn thận: nums[-1] = 4 (phần tử cuối, KHÔNG phải 'trước phần tử 0')


### 2.3. Kiểm tra mảng có tăng nghiêm ngặt không

Ta viết hàm nhỏ `is_strictly_increasing` để dùng trong test. `all(...)` trả `True` nếu **mọi** điều kiện bên trong đều đúng; với dãy rỗng (mảng 1 phần tử) `all` trả `True` — khớp với đề.

In [4]:
def is_strictly_increasing(arr):
    return all(arr[i] < arr[i + 1] for i in range(len(arr) - 1))

print(is_strictly_increasing([1, 2, 3]))   # True
print(is_strictly_increasing([1, 2, 2]))   # False: bằng nhau không được
print(is_strictly_increasing([8]))         # True: một phần tử

True
False
True


### 2.4. `max(a, b)` và `+=`

- `max(a, b)` trả số lớn hơn. Ta sẽ dùng `max(nums[i], prev + 1)`: "giữ nguyên nếu đã đủ lớn, nếu không thì nâng lên `prev + 1`".
- `ops += x` là viết tắt của `ops = ops + x`.

In [5]:
prev = 5
for x in [2, 6, 9]:
    print(f"x={x}, prev+1={prev + 1}, max(x, prev+1) = {max(x, prev + 1)}")

ops = 0
ops += 4
ops += 3
print("ops =", ops)

x=2, prev+1=6, max(x, prev+1) = 6
x=6, prev+1=6, max(x, prev+1) = 6
x=9, prev+1=6, max(x, prev+1) = 9
ops = 7


### 2.5. Sao chép list: `nums[:]` khác `b = nums`

Nếu lời giải sửa mảng tại chỗ, lúc test ta nên đưa **bản sao** để không làm hỏng dữ liệu gốc. `b = nums` chỉ tạo **tên mới cho cùng một list**; `nums[:]` tạo list mới.

In [6]:
nums = [1, 1, 1]
alias = nums        # cùng một list
copy = nums[:]      # list mới
alias[1] = 99
print("nums  =", nums)   # bị đổi theo alias
print("copy  =", copy)   # không bị ảnh hưởng

nums  = [1, 99, 1]
copy  = [1, 1, 1]


## 3. Cách trực giác — mô phỏng từng phép `+1`

Người mới thường nghĩ: "Cứ tìm chỗ vi phạm `nums[i] <= nums[i-1]` rồi tăng `nums[i]` lên 1, lặp đến khi hết vi phạm."

Cách này **đúng** (mỗi lần tăng đều cần thiết, xem mục 9), nhưng mỗi vòng ngoài chỉ thêm được 1 đơn vị.

In [7]:
def min_ops_simulation(nums: List[int]) -> int:
    arr = nums[:]          # không sửa input
    ops = 0
    changed = True
    while changed:
        changed = False
        for i in range(1, len(arr)):
            if arr[i] <= arr[i - 1]:
                arr[i] += 1
                ops += 1
                changed = True
    return ops

for nums, expected in examples:
    print(nums, "->", min_ops_simulation(nums), "| expected", expected)

[1, 1, 1] -> 3 | expected 3
[1, 5, 2, 4, 1] -> 14 | expected 14
[8] -> 0 | expected 0


### 3.1. Nút thắt của cách mô phỏng

Mỗi phép `+1` tốn ít nhất một lần duyệt. Với `nums = [10000, 1, 1, ..., 1]` (5000 phần tử), phần tử cuối cần tăng gần `10000 + 5000` đơn vị, và tổng số phép lên tới hàng chục triệu — quá chậm.

Ta đếm số lần quét mảng để thấy rõ chi phí.

In [8]:
def count_passes(nums):
    arr = nums[:]
    passes = 0
    changed = True
    while changed:
        changed = False
        passes += 1
        for i in range(1, len(arr)):
            if arr[i] <= arr[i - 1]:
                arr[i] += 1
                changed = True
    return passes

for nums in [[1, 1, 1], [1, 5, 2, 4, 1], [50, 1, 1, 1]]:
    print(nums, "-> số lần quét:", count_passes(nums), "| số phép:", min_ops_simulation(nums))

[1, 1, 1] -> số lần quét: 3 | số phép: 3
[1, 5, 2, 4, 1] -> số lần quét: 8 | số phép: 14
[50, 1, 1, 1] -> số lần quét: 53 | số phép: 153


**Câu hỏi then chốt:** thay vì tăng từng 1 đơn vị, ta có biết **ngay** `nums[i]` cần thành bao nhiêu không?

Có! Nếu phần tử trước (sau khi đã sửa) là `prev`, thì `nums[i]` phải **ít nhất** `prev + 1`. Và không có lý do gì để tăng **hơn** mức đó.

## 4. Insight — tham lam (greedy): nâng vừa đủ

**Greedy** = mỗi bước chọn lựa chọn "tốt nhất trước mắt" mà không quay lại. Ở đây:

- Nếu `nums[i] > prev`: giữ nguyên, tốn `0` phép, `prev = nums[i]`.
- Nếu `nums[i] <= prev`: nâng lên đúng `prev + 1`, tốn `prev + 1 - nums[i]` phép, `prev = prev + 1`.

Gộp lại: `new = max(nums[i], prev + 1)`, `ops += new - nums[i]`, `prev = new`.

Vì sao "vừa đủ" là tốt nhất? Giá trị `prev` càng **nhỏ** thì các phần tử sau càng **dễ** vượt qua. Tăng thừa chỉ làm tốn phép hiện tại **và** có thể làm tốn thêm phép ở sau.

In [9]:
# Tính tay ví dụ 2 bằng công thức greedy
nums = [1, 5, 2, 4, 1]
prev = nums[0]
for x in nums[1:]:
    new = max(x, prev + 1)
    print(f"x={x}: prev={prev} -> new=max({x},{prev + 1})={new}, tốn {new - x}")
    prev = new

x=5: prev=1 -> new=max(5,2)=5, tốn 0
x=2: prev=5 -> new=max(2,6)=6, tốn 4
x=4: prev=6 -> new=max(4,7)=7, tốn 3
x=1: prev=7 -> new=max(1,8)=8, tốn 7


### 4.1. Ví dụ: tăng thừa thì tệ hơn

So sánh hai cách xử lý `[1, 1, 3]`:

- Vừa đủ: `[1, 2, 3]` → tốn 1.
- Tăng thừa phần tử giữa lên 3: `[1, 3, 4]` → tốn 2 + 1 = 3.

Tăng thừa ở chỗ trước đẩy chi phí sang chỗ sau.

In [10]:
def cost_to(nums, target):
    return sum(t - x for t, x in zip(target, nums))

nums = [1, 1, 3]
print("vừa đủ  [1,2,3] tốn", cost_to(nums, [1, 2, 3]))
print("tăng thừa [1,3,4] tốn", cost_to(nums, [1, 3, 4]))

vừa đủ  [1,2,3] tốn 1
tăng thừa [1,3,4] tốn 3


`zip(a, b)` ghép từng cặp phần tử cùng vị trí: `zip([1,2],[5,6])` → `(1,5), (2,6)`. Ta dùng nó để cộng chênh lệch `target - nums` ở từng vị trí.

In [11]:
for t, x in zip([1, 2, 3], [1, 1, 3]):
    print("target", t, "- nums", x, "=", t - x)

target 1 - nums 1 = 0
target 2 - nums 1 = 1
target 3 - nums 3 = 0


## 5. Lời giải chính — giảng từng đoạn rồi ghép

**Đoạn 1 — khởi tạo:** phần tử đầu không bao giờ phải tăng, nên `prev = nums[0]`, `ops = 0`.

**Đoạn 2 — duyệt từ phần tử thứ hai:** `for x in nums[1:]` (hoặc `for i in range(1, n)`).

**Đoạn 3 — tính giá trị mới:** `new = max(x, prev + 1)`.

**Đoạn 4 — cộng chi phí và cập nhật:** `ops += new - x`, `prev = new`.

Cú pháp `nums[1:]` là **slicing**: lấy từ chỉ số 1 tới hết (tạo list mới, tốn `O(n)` bộ nhớ). Nếu muốn tiết kiệm, dùng `range(1, len(nums))`.

In [12]:
nums = [1, 5, 2, 4, 1]
print("nums[1:] =", nums[1:])
print("nums[:2] =", nums[:2])

nums[1:] = [5, 2, 4, 1]
nums[:2] = [1, 5]


In [13]:
def min_ops_greedy(nums: List[int]) -> int:
    prev = nums[0]            # phần tử đầu giữ nguyên
    ops = 0
    for i in range(1, len(nums)):
        x = nums[i]
        new = max(x, prev + 1)  # vừa đủ lớn hơn prev
        ops += new - x
        prev = new
    return ops

for nums, expected in examples:
    print(nums, "->", min_ops_greedy(nums), "| expected", expected)
    assert min_ops_greedy(nums) == expected

[1, 1, 1] -> 3 | expected 3
[1, 5, 2, 4, 1] -> 14 | expected 14
[8] -> 0 | expected 0


### 5.1. Biến thể cài đặt: sửa mảng tại chỗ

Gợi ý của LeetCode viết: `nums[i] = max(nums[i-1] + 1, nums[i])`. Lúc này `nums[i-1]` **đã là giá trị sau sửa**, đóng vai trò `prev`. Ta tính chênh lệch **trước** khi ghi đè.

In [14]:
def min_ops_inplace(nums: List[int]) -> int:
    ops = 0
    for i in range(1, len(nums)):
        if nums[i] <= nums[i - 1]:
            need = nums[i - 1] + 1
            ops += need - nums[i]
            nums[i] = need
    return ops

data = [1, 5, 2, 4, 1]
print("ops =", min_ops_inplace(data))
print("mảng sau khi sửa:", data, "| tăng nghiêm ngặt?", is_strictly_increasing(data))

ops = 14
mảng sau khi sửa: [1, 5, 6, 7, 8] | tăng nghiêm ngặt? True


## 6. TRACE từng bước

Bảng trace cho `nums = [1, 5, 2, 4, 1]`:

| i | x = nums[i] | prev (trước) | prev+1 | new = max(x, prev+1) | tốn | ops (sau) |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | 5 | 1 | 2 | 5 | 0 | 0 |
| 2 | 2 | 5 | 6 | 6 | 4 | 4 |
| 3 | 4 | 6 | 7 | 7 | 3 | 7 |
| 4 | 1 | 7 | 8 | 8 | 7 | 14 |

Ý nghĩa cột: `prev` là giá trị **cuối cùng** của phần tử đứng trước (đã sửa), không phải giá trị gốc.

In [15]:
def trace_min_ops(nums):
    prev = nums[0]
    ops = 0
    print(f"Bắt đầu: prev = nums[0] = {prev}, ops = 0")
    print(" i |  x | prev | prev+1 | new | tốn | ops")
    for i in range(1, len(nums)):
        x = nums[i]
        new = max(x, prev + 1)
        cost = new - x
        action = "giữ nguyên" if cost == 0 else f"nâng lên {new}"
        print(f"{i:>2} | {x:>2} | {prev:>4} | {prev + 1:>6} | {new:>3} | {cost:>3} | {ops + cost:>3}  <- {action}")
        ops += cost
        prev = new
    print("Kết quả ops =", ops)
    return ops

trace_min_ops([1, 5, 2, 4, 1])

Bắt đầu: prev = nums[0] = 1, ops = 0
 i |  x | prev | prev+1 | new | tốn | ops
 1 |  5 |    1 |      2 |   5 |   0 |   0  <- giữ nguyên
 2 |  2 |    5 |      6 |   6 |   4 |   4  <- nâng lên 6
 3 |  4 |    6 |      7 |   7 |   3 |   7  <- nâng lên 7
 4 |  1 |    7 |      8 |   8 |   7 |  14  <- nâng lên 8
Kết quả ops = 14


14

Trace thứ hai với `[1, 1, 1]` — nhánh "nâng lên" xảy ra liên tiếp, cho thấy `prev` đã bị nâng thì phần tử sau phải vượt **giá trị mới** chứ không phải giá trị gốc 1.

In [16]:
assert trace_min_ops([1, 1, 1]) == 3
print()
assert trace_min_ops([3, 2, 10, 1]) == 2 + 0 + 10

Bắt đầu: prev = nums[0] = 1, ops = 0
 i |  x | prev | prev+1 | new | tốn | ops
 1 |  1 |    1 |      2 |   2 |   1 |   1  <- nâng lên 2
 2 |  1 |    2 |      3 |   3 |   2 |   3  <- nâng lên 3
Kết quả ops = 3

Bắt đầu: prev = nums[0] = 3, ops = 0
 i |  x | prev | prev+1 | new | tốn | ops
 1 |  2 |    3 |      4 |   4 |   2 |   2  <- nâng lên 4
 2 | 10 |    4 |      5 |  10 |   0 |   2  <- giữ nguyên
 3 |  1 |   10 |     11 |  11 |  10 |  12  <- nâng lên 11
Kết quả ops = 12


## 7. Cách sáng tạo — công thức prefix max của `nums[j] - j`

Gọi `t[i]` là giá trị cuối cùng theo greedy. Quan sát: `t[i] = max(nums[i], t[i-1] + 1)`. Trừ cả hai vế cho `i`:

`t[i] - i = max(nums[i] - i, t[i-1] - (i-1))`.

Đặt `b[i] = nums[i] - i`, ta có `t[i] - i = max(b[0], ..., b[i])` — **prefix max** (giá trị lớn nhất tính từ đầu tới `i`). Vậy:

`ops = Σ (t[i] - nums[i]) = Σ (prefixmax_b[i] - b[i])`.

Ý nghĩa: "trừ đi `i`" biến điều kiện **tăng nghiêm ngặt** thành **không giảm**, và để một dãy không giảm với chỉ phép tăng, mỗi phần tử phải lên tới max của những gì đứng trước.

In [17]:
nums = [1, 5, 2, 4, 1]
b = [x - i for i, x in enumerate(nums)]
print("b = nums[i] - i =", b)
pm = []
cur = float("-inf")
for v in b:
    cur = max(cur, v)
    pm.append(cur)
print("prefix max      =", pm)
print("t = pm + i      =", [p + i for i, p in enumerate(pm)])
print("ops             =", sum(p - v for p, v in zip(pm, b)))

b = nums[i] - i = [1, 4, 0, 1, -3]
prefix max      = [1, 4, 4, 4, 4]
t = pm + i      = [1, 5, 6, 7, 8]
ops             = 14


`enumerate(nums)` sinh cặp `(chỉ_số, giá_trị)`. `float("-inf")` là "âm vô cùng", nhỏ hơn mọi số — tiện làm giá trị khởi đầu cho max.

In [18]:
for i, x in enumerate([7, 8, 9]):
    print("i =", i, "x =", x)
print(float("-inf") < -10**9)

i = 0 x = 7
i = 1 x = 8
i = 2 x = 9
True


In [19]:
from itertools import accumulate

def min_ops_prefix_max(nums: List[int]) -> int:
    b = [x - i for i, x in enumerate(nums)]
    pm = list(accumulate(b, max))   # prefix max
    return sum(p - v for p, v in zip(pm, b))

print(list(accumulate([1, 4, 0, 1, -3], max)))   # accumulate với max = prefix max
for nums, expected in examples:
    assert min_ops_prefix_max(nums) == expected
print("prefix max PASS")

[1, 4, 4, 4, 4]
prefix max PASS


`itertools.accumulate(seq, f)` tính dồn: phần tử thứ k là `f(... f(f(a0, a1), a2) ..., ak)`. Với `f = max` ta được prefix max; với mặc định (cộng) ta được prefix sum.

## 8. Code nộp LeetCode

### Cách A — greedy với biến `prev` (khuyên dùng)

In [20]:
class Solution:
    def minOperations(self, nums: List[int]) -> int:
        prev = nums[0]
        ops = 0
        for i in range(1, len(nums)):
            new = max(nums[i], prev + 1)
            ops += new - nums[i]
            prev = new
        return ops

sol = Solution()
print(sol.minOperations([1, 1, 1]), sol.minOperations([1, 5, 2, 4, 1]), sol.minOperations([8]))

3 14 0


### Cách B — sửa tại chỗ theo gợi ý đề

In [21]:
class SolutionInPlace:
    def minOperations(self, nums: List[int]) -> int:
        ops = 0
        for i in range(1, len(nums)):
            if nums[i] <= nums[i - 1]:
                ops += nums[i - 1] + 1 - nums[i]
                nums[i] = nums[i - 1] + 1
        return ops

print(SolutionInPlace().minOperations([1, 5, 2, 4, 1]))

14


### Cách C — prefix max (một dòng tư duy khác)

In [22]:
class SolutionPrefixMax:
    def minOperations(self, nums: List[int]) -> int:
        best = float("-inf")
        ops = 0
        for i, x in enumerate(nums):
            best = max(best, x - i)
            ops += best - (x - i)
        return ops

print(SolutionPrefixMax().minOperations([1, 5, 2, 4, 1]))

14


## 9. Vì sao greedy đúng? (invariant)

Gọi `t` là bất kỳ mảng kết quả hợp lệ nào (`t[i] >= nums[i]`, tăng nghiêm ngặt). Chi phí là `Σ (t[i] - nums[i])`.

**Bổ đề:** với mọi `i`, `t[i] >= g[i]`, trong đó `g` là mảng greedy.

- `i = 0`: `g[0] = nums[0] <= t[0]`.
- Giả sử `t[i-1] >= g[i-1]`. Vì `t` tăng nghiêm ngặt: `t[i] >= t[i-1] + 1 >= g[i-1] + 1`. Và `t[i] >= nums[i]`. Nên `t[i] >= max(nums[i], g[i-1] + 1) = g[i]`.

Do đó `Σ t[i] >= Σ g[i]`, tức **mọi** lời giải hợp lệ đều tốn ít nhất bằng greedy. Mà greedy cũng hợp lệ ⇒ greedy **tối ưu**.

**Invariant khi chạy vòng lặp:** sau khi xử lý xong chỉ số `i`, `prev` là giá trị **nhỏ nhất có thể** của phần tử `i` trong mọi cách sửa hợp lệ của đoạn `nums[0..i]`, và `ops` là chi phí nhỏ nhất cho đoạn đó.

Bổ đề này cũng giải thích vì sao mô phỏng `+1` ở mục 3 đúng: nó không bao giờ đẩy phần tử nào vượt `g[i]`.

In [23]:
# Kiểm chứng bổ đề bằng vét cạn trên mảng nhỏ: thử mọi mảng t hợp lệ
from itertools import product

def min_ops_bruteforce(nums, extra=6):
    best = None
    ranges = [range(x, x + extra + len(nums)) for x in nums]
    for t in product(*ranges):
        if is_strictly_increasing(t):
            c = sum(a - b for a, b in zip(t, nums))
            if best is None or c < best:
                best = c
    return best

for nums in [[1, 1, 1], [3, 1, 2], [2, 2, 1, 1], [1, 4, 1]]:
    print(nums, "vét cạn =", min_ops_bruteforce(nums), "| greedy =", min_ops_greedy(nums))
    assert min_ops_bruteforce(nums) == min_ops_greedy(nums)

[1, 1, 1] vét cạn = 3 | greedy = 3
[3, 1, 2] vét cạn = 6 | greedy = 6
[2, 2, 1, 1] vét cạn = 8 | greedy = 8
[1, 4, 1] vét cạn = 4 | greedy = 4


`itertools.product(*ranges)` sinh **mọi tổ hợp** chọn một giá trị từ mỗi range (tích Descartes). Dấu `*` "trải" list ranges thành các tham số riêng. Cách này bùng nổ rất nhanh, chỉ dùng làm "oracle" kiểm tra trên input tí hon.

## 10. Độ phức tạp

Gọi `n = len(nums)`, `S` = đáp án (tổng số phép).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Mô phỏng `+1` | `O(n · (S + 1))` xấu nhất | `O(n)` (bản sao) | quá chậm khi S lớn |
| Greedy `prev` | `O(n)` | `O(1)` | khuyên dùng |
| Greedy tại chỗ | `O(n)` | `O(1)` | sửa input |
| Prefix max (bản list) | `O(n)` | `O(n)` | bản vòng lặp chỉ `O(1)` |
| Vét cạn `product` | mũ | — | chỉ để kiểm thử |

Đáp án có thể tới cỡ `6·10^7` (khoảng `5000 · (10^4 + 2500)`, xem cell dưới) — vẫn vừa kiểu `int` 32-bit (≈ 2.1·10^9) của C++/Java; Python int thì không bao giờ tràn.

In [24]:
worst = [10**4] + [1] * 4999
print("đáp án trường hợp xấu:", min_ops_greedy(worst))

đáp án trường hợp xấu: 62482501


## 11. Bộ test

Nhóm test: mẫu đề; một phần tử; đã tăng sẵn (0 phép); toàn bằng nhau; giảm dần; có phần tử lớn ở đầu; ngẫu nhiên đối chiếu nhiều cách.

In [25]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS ({actual})")

check("mẫu 1", min_ops_greedy([1, 1, 1]), 3)
check("mẫu 2", min_ops_greedy([1, 5, 2, 4, 1]), 14)
check("mẫu 3", min_ops_greedy([8]), 0)
check("đã tăng sẵn", min_ops_greedy([1, 2, 3, 10]), 0)
check("có bằng nhau", min_ops_greedy([1, 2, 2]), 1)
check("giảm dần", min_ops_greedy([4, 3, 2, 1]), 2 + 4 + 6)
check("đỉnh ở đầu", min_ops_greedy([100, 1]), 100)
check("toàn 5", min_ops_greedy([5] * 4), 0 + 1 + 2 + 3)

mẫu 1                  PASS (3)
mẫu 2                  PASS (14)
mẫu 3                  PASS (0)
đã tăng sẵn            PASS (0)
có bằng nhau           PASS (1)
giảm dần               PASS (12)
đỉnh ở đầu             PASS (100)
toàn 5                 PASS (6)


In [26]:
import random
rng = random.Random(1827)
for _ in range(500):
    n = rng.randint(1, 8)
    nums = [rng.randint(1, 10) for _ in range(n)]
    g = min_ops_greedy(nums)
    assert g == min_ops_simulation(nums) == min_ops_prefix_max(nums) == Solution().minOperations(nums[:])
    assert g == SolutionPrefixMax().minOperations(nums) == SolutionInPlace().minOperations(nums[:])
for _ in range(60):
    nums = [rng.randint(1, 4) for _ in range(rng.randint(1, 4))]
    assert min_ops_bruteforce(nums) == min_ops_greedy(nums)
print("Đối chiếu ngẫu nhiên PASS")

Đối chiếu ngẫu nhiên PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: so với giá trị **gốc** thay vì giá trị đã sửa

In [27]:
def wrong_compare_original(nums):
    ops = 0
    for i in range(1, len(nums)):
        if nums[i] <= nums[i - 1]:          # nums[i-1] là giá trị gốc!
            ops += nums[i - 1] + 1 - nums[i]
    return ops

print("sai :", wrong_compare_original([1, 1, 1]), "| đúng:", min_ops_greedy([1, 1, 1]))

sai : 2 | đúng: 3


Vì sao sai: phần tử giữa đã được nâng lên 2, nên phần tử cuối phải vượt **2** chứ không phải 1. Sửa: lưu `prev` (hoặc ghi đè `nums[i]`).

### Lỗi 2: dùng `<` thay vì `<=` (quên "nghiêm ngặt")

In [28]:
def wrong_non_strict(nums):
    prev, ops = nums[0], 0
    for x in nums[1:]:
        new = max(x, prev)          # quên +1
        ops += new - x
        prev = new
    return ops

print("sai :", wrong_non_strict([1, 1, 1]), "| đúng:", min_ops_greedy([1, 1, 1]))

sai : 0 | đúng: 3


Kết quả 0 là của bài "không giảm". Sửa: `max(x, prev + 1)`.

### Lỗi 3: ghi đè trước rồi mới tính chi phí

In [29]:
def wrong_overwrite_first(nums):
    nums = nums[:]
    ops = 0
    for i in range(1, len(nums)):
        if nums[i] <= nums[i - 1]:
            nums[i] = nums[i - 1] + 1
            ops += nums[i - 1] + 1 - nums[i]   # nums[i] đã bị đổi -> luôn 0
    return ops

print("sai :", wrong_overwrite_first([1, 5, 2, 4, 1]), "| đúng:", min_ops_greedy([1, 5, 2, 4, 1]))

sai : 0 | đúng: 14


Sửa: tính `ops += need - nums[i]` **trước**, rồi mới `nums[i] = need`.

### Lỗi 4: duyệt từ `i = 0`

`nums[0 - 1]` là `nums[-1]` = phần tử **cuối**, Python không báo lỗi nên rất khó phát hiện.

In [30]:
def wrong_start_zero(nums):
    nums = nums[:]
    ops = 0
    for i in range(0, len(nums)):
        if nums[i] <= nums[i - 1]:
            ops += nums[i - 1] + 1 - nums[i]
            nums[i] = nums[i - 1] + 1
    return ops

print("sai :", wrong_start_zero([1, 2, 3]), "| đúng:", min_ops_greedy([1, 2, 3]))

sai : 9 | đúng: 0


## 13. Mẹo nhận dạng pattern

- Đề cho phép **chỉ tăng** (hoặc chỉ giảm) và yêu cầu **ít phép nhất** để thỏa một thứ tự ⇒ nghĩ ngay tới **greedy một chiều**: đi trái → phải, giữ giá trị **nhỏ nhất có thể** của phần tử vừa xử lý.
- Từ khóa "strictly increasing" + "increment" ⇒ công thức `max(x, prev + 1)`.
- Mẹo "trừ chỉ số": `nums[i] - i` biến "tăng nghiêm ngặt" thành "không giảm" — hữu ích ở nhiều bài khác.
- Luôn tự hỏi: "giữ biến `prev` là giá trị **gốc** hay **đã sửa**?" — đây là nguồn lỗi số 1.

## 14. Biến thể

1. **Không giảm** (`<=`) thay vì tăng nghiêm ngặt: `new = max(x, prev)`.
2. **Cách nhau ít nhất `k`**: `new = max(x, prev + k)`.
3. **LeetCode 945 — Minimum Increment to Make Array Unique**: sắp xếp trước, rồi áp dụng đúng greedy này (thứ tự tự do nên sort là hợp lệ).
4. Trả về **mảng kết quả** thay vì số phép.

In [31]:
def min_ops_gap_k(nums, k):
    prev, ops = nums[0], 0
    for x in nums[1:]:
        new = max(x, prev + k)
        ops += new - x
        prev = new
    return ops

def min_increment_unique(nums):
    if not nums:
        return 0
    return min_ops_greedy(sorted(nums))

print("gap k=1 (chính bài):", min_ops_gap_k([1, 5, 2, 4, 1], 1))
print("gap k=2:", min_ops_gap_k([1, 1, 1], 2))           # [1,3,5] -> 2+4
print("LC945 [3,2,1,2,1,7]:", min_increment_unique([3, 2, 1, 2, 1, 7]))  # 6

gap k=1 (chính bài): 14
gap k=2: 6
LC945 [3,2,1,2,1,7]: 6


`sorted(nums)` trả về **list mới** đã sắp xếp tăng dần (không đổi `nums`), khác `nums.sort()` sửa tại chỗ và trả `None`.

## 15. Bài tập (tự làm, chưa có đáp án ở đây)

**Bài 1 — tính tay:** với `nums = [2, 2, 5, 3, 3]`, lập bảng trace (`x`, `prev`, `new`, tốn) và cho đáp án.

**Bài 2 — dự đoán output:** cell dưới in gì? (đừng chạy trước khi đoán)

```python
prev = 4
for x in [1, 9, 9]:
    prev = max(x, prev + 1)
    print(prev, end=" ")
```

**Bài 3 — điền TODO:** hoàn thiện greedy bằng biến `prev`.

In [32]:
def exercise_min_ops(nums):
    # TODO 1: prev = phần tử đầu, ops = 0
    # TODO 2: duyệt từ chỉ số 1
    # TODO 3: new = max(x, prev + 1); cộng chi phí; cập nhật prev
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_min_ops([1, 1, 1]) == 3
# assert exercise_min_ops([1, 5, 2, 4, 1]) == 14

**Bài 4 — trả về mảng kết quả:** viết `exercise_final_array(nums)` trả về mảng tăng nghiêm ngặt có chi phí nhỏ nhất (không sửa input).

In [33]:
def exercise_final_array(nums):
    # TODO: giống greedy nhưng append new vào list kết quả
    pass

# assert exercise_final_array([1, 5, 2, 4, 1]) == [1, 5, 6, 7, 8]

**Bài 5 — biến thể chỉ được giảm:** mỗi phép chọn một phần tử và **giảm** nó đi 1. Tìm số phép ít nhất để mảng **tăng nghiêm ngặt**, biết các phần tử có thể âm. (Gợi ý hướng đi: duyệt từ **phải sang trái**.)

In [34]:
def exercise_min_decrements(nums):
    # TODO: duyệt ngược, giữ nxt = giá trị đã sửa của phần tử bên phải
    pass

# assert exercise_min_decrements([3, 3, 3]) == 3     # -> [1, 2, 3]
# assert exercise_min_decrements([1, 2, 3]) == 0

**Bài 6 — mini challenge (LC 945):** cho mảng **không có thứ tự cố định**, mỗi phép tăng một phần tử 1 đơn vị; tìm số phép ít nhất để mọi phần tử **khác nhau**.

In [35]:
def exercise_min_increment_unique(nums):
    # TODO: sắp xếp rồi áp dụng greedy prev + 1
    pass

# assert exercise_min_increment_unique([1, 2, 2]) == 1
# assert exercise_min_increment_unique([3, 2, 1, 2, 1, 7]) == 6

## 16. Gợi ý 3 mức

| Bài | Mức 1 (hướng) | Mức 2 (cấu trúc) | Mức 3 (gần code) |
|---|---|---|---|
| 1 | phần tử đầu giữ nguyên | so với `prev` đã sửa | `new` lần lượt 2,3,5,6,7 |
| 2 | `prev` cập nhật mỗi vòng | `max(1,5)`, rồi `max(9,6)`… | in 3 số cách nhau dấu cách |
| 3 | một biến `prev` | `for i in range(1, len(nums))` | `new = max(nums[i], prev + 1)` |
| 4 | lưu lại `new` | `res = [nums[0]]` | `res.append(new)` |
| 5 | giảm thì nên sửa từ phải | `nxt = nums[-1]` | `new = min(x, nxt - 1)` |
| 6 | thứ tự không quan trọng | `sorted(nums)` | rồi chạy greedy bài chính |

## 17. Đáp án tham khảo

**Bài 1:** `[2,2,5,3,3]` → `new` = 2, 3, 5, 6, 7 → tốn 0+1+0+3+4 = **8**.

**Bài 2:** in `5 9 10`.

In [36]:
prev = 4
for x in [1, 9, 9]:
    prev = max(x, prev + 1)
    print(prev, end=" ")
print()
assert min_ops_greedy([2, 2, 5, 3, 3]) == 8

5 9 10 


In [37]:
def answer_min_ops(nums):
    prev, ops = nums[0], 0
    for i in range(1, len(nums)):
        new = max(nums[i], prev + 1)
        ops += new - nums[i]
        prev = new
    return ops


def answer_final_array(nums):
    res = [nums[0]]
    for x in nums[1:]:
        res.append(max(x, res[-1] + 1))
    return res


def answer_min_decrements(nums):
    nxt = nums[-1]
    ops = 0
    for i in range(len(nums) - 2, -1, -1):
        new = min(nums[i], nxt - 1)
        ops += nums[i] - new
        nxt = new
    return ops


def answer_min_increment_unique(nums):
    arr = sorted(nums)
    ops = 0
    for i in range(1, len(arr)):
        if arr[i] <= arr[i - 1]:
            ops += arr[i - 1] + 1 - arr[i]
            arr[i] = arr[i - 1] + 1
    return ops


assert answer_min_ops([1, 1, 1]) == 3
assert answer_min_ops([1, 5, 2, 4, 1]) == 14
assert answer_final_array([1, 5, 2, 4, 1]) == [1, 5, 6, 7, 8]
assert answer_min_decrements([3, 3, 3]) == 3
assert answer_min_decrements([1, 2, 3]) == 0
assert answer_min_increment_unique([1, 2, 2]) == 1
assert answer_min_increment_unique([3, 2, 1, 2, 1, 7]) == 6
for _ in range(200):
    nums = [rng.randint(1, 9) for _ in range(rng.randint(1, 7))]
    fa = answer_final_array(nums)
    assert is_strictly_increasing(fa) and sum(fa) - sum(nums) == answer_min_ops(nums)
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


`range(len(nums) - 2, -1, -1)` đếm **ngược**: bắt đầu ở chỉ số áp chót, dừng **trước** `-1` (tức dừng ở 0), bước `-1`.

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def minOperations(self, nums: List[int]) -> int:
        prev, ops = nums[0], 0
        for i in range(1, len(nums)):
            new = max(nums[i], prev + 1)   # vừa đủ lớn hơn phần tử trước (đã sửa)
            ops += new - nums[i]
            prev = new
        return ops
```

Công thức nhớ: **"nâng vừa đủ: `new = max(x, prev + 1)`"**.

Checklist trước khi submit:

- [ ] Duyệt từ chỉ số **1**, không phải 0.
- [ ] So với giá trị **đã sửa** (`prev`), không phải giá trị gốc.
- [ ] Dùng `prev + 1` (nghiêm ngặt), không phải `prev`.
- [ ] Nếu sửa tại chỗ: cộng chi phí **trước** khi ghi đè.
- [ ] Mảng 1 phần tử trả `0`.
- [ ] Đã import `List` (`from typing import List`) nếu chạy ngoài LeetCode.